# Riesgo de crédito en un banco
## Notebook 03 · Análisis exploratorio del riesgo

**Entrada:** tabla `prestamos_variables` de `data/banco.db`, generada en `02_variables_sql.ipynb`.

**Qué hace este notebook:** compara la tasa de mora entre grupos de préstamos para identificar los factores de riesgo, antes de construir un modelo.

**Conceptos estadísticos:**
- **Tasa de mora de un grupo** = probabilidad condicional P(malo | grupo).
- **Lift** = cuántas veces más (o menos) riesgo tiene un grupo que el promedio.
- **Intervalo de confianza:** con pocos préstamos, una tasa puede variar mucho por azar. El intervalo muestra ese margen.

In [2]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

conexion = sqlite3.connect("../data/banco.db")

def consulta(sql):
    return pd.read_sql(sql, conexion)

# Columnas de las tablas del cliente y del distrito (para revisar sus nombres)
print("client:", consulta("SELECT * FROM client LIMIT 1").columns.tolist())
print("district:", consulta("SELECT * FROM district LIMIT 1").columns.tolist())

client: ['client_id', 'gender', 'birth_date', 'district_id']
district: ['district_id', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']


In [3]:
df = consulta("""
SELECT
    v.*,
    c.gender                                                AS genero,
    ROUND((julianday(v.fecha_prestamo) - julianday(c.birth_date)) / 365.25, 0) AS edad,
    d.A11                                                   AS salario_promedio_distrito,
    d.A13                                                   AS desempleo_distrito
FROM prestamos_variables AS v
JOIN disp AS dp
    ON  dp.account_id = v.account_id
    AND dp.type = 'Titular'
JOIN client AS c
    ON c.client_id = dp.client_id
JOIN district AS d
    ON d.district_id = c.district_id
""")

assert len(df) == 682, "El JOIN duplicó o perdió préstamos"

tasa_global = df["malo"].mean()
print(f"Préstamos: {len(df)} | Tasa de mora global: {tasa_global:.1%}")

Préstamos: 682 | Tasa de mora global: 11.1%


---
## 1. Tasa de mora por grupo

Cada variable numérica se divide en **cuartiles** (cuatro grupos con el mismo número de préstamos) y se calcula la tasa de mora de cada uno.

Para cada tasa se calcula un **intervalo de confianza del 95% (método de Wilson)**. Con grupos de unos 170 préstamos y pocos casos de mora, una diferencia de 2 o 3 puntos puede deberse solo al azar. Si los intervalos de dos grupos se superponen mucho, no se puede afirmar que su riesgo sea distinto.

In [4]:
def intervalo_wilson(malos, n, z=1.96):
    """Intervalo de confianza del 95% para una proporción (método de Wilson)."""
    p = malos / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - margen, centro + margen


def tasa_por_grupo(datos, columna, cuantiles=4):
    """Tasa de mora, intervalo de confianza y lift por grupo de una variable."""
    grupos = pd.qcut(datos[columna], q=cuantiles, duplicates="drop")
    tabla = datos.groupby(grupos, observed=True)["malo"].agg(prestamos="count", malos="sum")
    tabla["tasa"] = tabla["malos"] / tabla["prestamos"]
    tabla[["ic_inf", "ic_sup"]] = [
        intervalo_wilson(m, n) for m, n in zip(tabla["malos"], tabla["prestamos"])
    ]
    tabla["lift"] = tabla["tasa"] / tasa_global
    return tabla.round(3)

In [5]:
variables_numericas = [
    "monto", "plazo_meses", "cuota", "meses_historial", "saldo_promedio",
    "saldo_minimo", "saldo_final", "ingreso_promedio", "edad", "salario_promedio_distrito",
]

for var in variables_numericas:
    print(f"\n===== {var} =====")
    display(tasa_por_grupo(df, var))


===== monto =====


,prestamos,malos,tasa,ic_inf,ic_sup,lift
monto,,,,,,
"(4979.999, 66732.0]",171,13,0.076,0.045,0.126,0.682
"(66732.0, 116928.0]",170,13,0.076,0.045,0.126,0.686
"(116928.0, 210654.0]",170,16,0.094,0.059,0.147,0.845
"(210654.0, 590820.0]",171,34,0.199,0.146,0.265,1.784



===== plazo_meses =====


,prestamos,malos,tasa,ic_inf,ic_sup,lift
plazo_meses,,,,,,
"(11.999, 24.0]",269,28,0.104,0.073,0.146,0.934
"(24.0, 36.0]",130,15,0.115,0.071,0.182,1.035
"(36.0, 48.0]",138,16,0.116,0.073,0.180,1.040
"(48.0, 60.0]",145,17,0.117,0.075,0.180,1.052



===== cuota =====


,prestamos,malos,tasa,ic_inf,ic_sup,lift
cuota,,,,,,
"(303.999, 2477.0]",172,10,0.058,0.032,0.104,0.522
"(2477.0, 3934.0]",169,14,0.083,0.050,0.134,0.743
"(3934.0, 5813.5]",170,19,0.112,0.073,0.168,1.003
"(5813.5, 9910.0]",171,33,0.193,0.141,0.259,1.732


... (10 outputs left out)


===== edad =====


,prestamos,malos,tasa,ic_inf,ic_sup,lift
edad,,,,,,
"(13.999, 27.0]",174,20,0.115,0.076,0.171,1.031
"(27.0, 38.0]",177,19,0.107,0.070,0.162,0.963
"(38.0, 49.0]",168,19,0.113,0.074,0.170,1.015
"(49.0, 62.0]",163,18,0.110,0.071,0.168,0.991



===== salario_promedio_distrito =====


,prestamos,malos,tasa,ic_inf,ic_sup,lift
salario_promedio_distrito,,,,,,
"(8109.999, 8544.0]",178,20,0.112,0.074,0.167,1.008
"(8544.0, 8980.0]",169,19,0.112,0.073,0.169,1.009
"(8980.0, 9897.0]",175,21,0.120,0.080,0.176,1.077
"(9897.0, 12541.0]",160,16,0.100,0.062,0.156,0.897


---
## 2. Señales de alarma: saldo negativo e intereses sancionatorios

Casi todos los préstamos tienen 0 en `n_saldo_negativo` y `n_interes_sancion`, así que no sirven los cuartiles. Se convierten en variables de sí/no y se comparan dos probabilidades:

- **P(malo | señal):** de los préstamos *con* la señal, ¿cuántos cayeron en mora?
- **P(señal | malo):** de los préstamos *malos*, ¿cuántos tenían la señal? (qué tan frecuente es la señal entre los malos)

No son lo mismo. Es la misma diferencia del teorema de Bayes que viste con las alertas de fraude.

In [6]:
for senal in ["n_saldo_negativo", "n_interes_sancion"]:
    tiene = df[senal] > 0
    malos = df["malo"] == 1

    p_malo_dado_senal = malos[tiene].mean()
    p_malo_sin_senal = malos[~tiene].mean()
    p_senal_dado_malo = tiene[malos].mean()

    print(f"\n===== {senal} > 0 =====")
    print(f"Préstamos con la señal: {tiene.sum()} de {len(df)}")
    print(f"P(malo | con señal) = {p_malo_dado_senal:.1%}")
    print(f"P(malo | sin señal) = {p_malo_sin_senal:.1%}")
    print(f"P(señal | malo)     = {p_senal_dado_malo:.1%}")


===== n_saldo_negativo > 0 =====
Préstamos con la señal: 27 de 682
P(malo | con señal) = 100.0%
P(malo | sin señal) = 7.5%
P(señal | malo)     = 35.5%

===== n_interes_sancion > 0 =====
Préstamos con la señal: 29 de 682
P(malo | con señal) = 89.7%
P(malo | sin señal) = 7.7%
P(señal | malo)     = 34.2%
